In [1]:
#NODE SNAPPING- KDTree
import json
import time
import numpy as np
from scipy.spatial import KDTree

start_time = time.time()

with open("tokyo_full_graph_updated.json") as f:
    graph = json.load(f)

print("Graph loaded")

nodes = graph["nodes"]

# Separate nodes
road_nodes = [n for n in nodes if n.get("type") is None]
poi_nodes = [n for n in nodes if n.get("type") in ["building", "hospital", "fire_station"]]

print(f"Road nodes: {len(road_nodes)}")
print(f"POI nodes: {len(poi_nodes)}")

# Build KDTree
road_coords = [(n["lat"], n["lon"]) for n in road_nodes]
tree = KDTree(road_coords)

print("KDTree built")

new_edges = []
distances = []

for i, poi in enumerate(poi_nodes):

    if i % 500 == 0:
        print(f"Processing {i}/{len(poi_nodes)}")

    dists, idxs = tree.query((poi["lat"], poi["lon"]), k=3)

    for d, idx in zip(dists, idxs):
        nearest_node = road_nodes[idx]

        distances.append(d)

        new_edges.append({
            "from": poi["id"],
            "to": nearest_node["id"],
            "length": d * 100000,
            "road_type": "connector_node"
        })

graph["edges"].extend(new_edges)

with open("tokyo_graph_node.json", "w") as f:
    json.dump(graph, f)

print("\n=== NODE KDTree METRICS ===")
print("Connections:", len(new_edges))
print("Avg distance:", sum(distances)/len(distances))
print("Max distance:", max(distances))
print("Min distance:", min(distances))

print(f"\nTime taken: {time.time() - start_time:.2f}s")

Graph loaded
Road nodes: 76609
POI nodes: 10482
KDTree built
Processing 0/10482
Processing 500/10482
Processing 1000/10482
Processing 1500/10482
Processing 2000/10482
Processing 2500/10482
Processing 3000/10482
Processing 3500/10482
Processing 4000/10482
Processing 4500/10482
Processing 5000/10482
Processing 5500/10482
Processing 6000/10482
Processing 6500/10482
Processing 7000/10482
Processing 7500/10482
Processing 8000/10482
Processing 8500/10482
Processing 9000/10482
Processing 9500/10482
Processing 10000/10482

=== NODE KDTree METRICS ===
Connections: 31446
Avg distance: 0.0002954127426577106
Max distance: 0.007120755320904533
Min distance: 0.0

Time taken: 2.54s


In [2]:
#EDGE SNAPPING (APPROX)- KDtree
import json
import time
import numpy as np
from scipy.spatial import KDTree
from collections import defaultdict

start_time = time.time()

with open("tokyo_full_graph_updated.json") as f:
    graph = json.load(f)

print("Graph loaded")

nodes = graph["nodes"]
edges = graph["edges"]

node_dict = {n["id"]: n for n in nodes}

poi_nodes = [n for n in nodes if n.get("type") in ["building", "hospital", "fire_station"]]

# Build KDTree on road nodes
road_nodes = [n for n in nodes if n.get("type") is None]
road_coords = [(n["lat"], n["lon"]) for n in road_nodes]
tree = KDTree(road_coords)

print("KDTree built")

# Build adjacency for edge lookup
adj = defaultdict(list)
for e in edges:
    adj[e["from"]].append(e)
    adj[e["to"]].append(e)

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return ((px-x1)**2 + (py-y1)**2)**0.5

    t = ((px-x1)*dx + (py-y1)*dy) / (dx*dx + dy*dy)
    t = max(0, min(1, t))

    proj_x = x1 + t*dx
    proj_y = y1 + t*dy

    return ((px-proj_x)**2 + (py-proj_y)**2)**0.5

new_edges = []
distances = []

for i, poi in enumerate(poi_nodes):

    if i % 500 == 0:
        print(f"Processing {i}/{len(poi_nodes)}")

    _, idxs = tree.query((poi["lat"], poi["lon"]), k=3)

    candidate_edges = set()

    for idx in idxs:
        node = road_nodes[idx]
        for e in adj[node["id"]]:
            candidate_edges.add((e["from"], e["to"]))

    px, py = poi["lon"], poi["lat"]

    edge_distances = []

    for edge in candidate_edges:
        n1 = node_dict.get(edge[0])
        n2 = node_dict.get(edge[1])
        if not n1 or not n2:
            continue

        d = point_to_segment_dist(px, py, n1["lon"], n1["lat"], n2["lon"], n2["lat"])
        edge_distances.append((d, {"from": edge[0], "to": edge[1]}))

    # sort edges by distance
    edge_distances.sort(key=lambda x: x[0])

    # ⚠️ FIX 1: Safety check — skip POI if no candidate edges found (sparse area)
    if not edge_distances:
        continue

    # take top 3 closest edges
    for d, edge in edge_distances[:3]:
        distances.append(d)

        # ✅ FIX 2: Connect to BOTH endpoints for bidirectional road coverage
        new_edges.append({
            "from": poi["id"],
            "to": edge["from"],
            "length": d * 100000,
            "road_type": "connector_edge"
        })

        new_edges.append({
            "from": poi["id"],
            "to": edge["to"],
            "length": d * 100000,
            "road_type": "connector_edge"
        })

graph["edges"].extend(new_edges)

with open("tokyo_graph_edge.json", "w") as f:
    json.dump(graph, f)

print("\n=== EDGE KDTree METRICS ===")
print("Connections:", len(new_edges))
print("Avg distance:", sum(distances)/len(distances))
print("Max distance:", max(distances))
print("Min distance:", min(distances))

print(f"\nTime taken: {time.time() - start_time:.2f}s")

Graph loaded
KDTree built
Processing 0/10482
Processing 500/10482
Processing 1000/10482
Processing 1500/10482
Processing 2000/10482
Processing 2500/10482
Processing 3000/10482
Processing 3500/10482
Processing 4000/10482
Processing 4500/10482
Processing 5000/10482
Processing 5500/10482
Processing 6000/10482
Processing 6500/10482
Processing 7000/10482
Processing 7500/10482
Processing 8000/10482
Processing 8500/10482
Processing 9000/10482
Processing 9500/10482
Processing 10000/10482

=== EDGE KDTree METRICS ===
Connections: 62892
Avg distance: 1.956531800444416e-05
Max distance: 0.0024315759201526165
Min distance: 0.0

Time taken: 3.42s


In [ ]:
#NODE SNAPPING- brute force- same results as kdtree but much slower- ignore
"""
import json
import math
import time

start_time = time.time()

with open("tokyo_full_graph_updated.json") as f:
    graph = json.load(f)

print("Graph loaded")

nodes = graph["nodes"]
edges = graph["edges"]

road_nodes = [n for n in nodes if n.get("type") is None]
poi_nodes = [n for n in nodes if n.get("type") in ["building", "hospital", "fire_station"]]

print(f"Total nodes: {len(nodes)}")
print(f"Road nodes: {len(road_nodes)}")
print(f"POI nodes: {len(poi_nodes)}")

def distance(n1, n2):
    return ((n1["lat"] - n2["lat"])**2 + (n1["lon"] - n2["lon"])**2)**0.5

new_edges = []
distances = []

# 🔥 IMPORTANT: process in chunks
for i, poi in enumerate(poi_nodes):

    # progress print every 500
    if i % 500 == 0:
        elapsed = time.time() - start_time
        print(f"Processing POI {i}/{len(poi_nodes)} | Time: {elapsed:.1f}s")

    nearest = None
    min_dist = float("inf")

    for road in road_nodes:
        d = distance(poi, road)
        if d < min_dist:
            min_dist = d
            nearest = road

    if nearest:
        distances.append(min_dist)
        new_edges.append({
            "from": poi["id"],
            "to": nearest["id"],
            "length": min_dist * 100000,
            "road_type": "connector_node"
        })

print("\nFinished connecting POIs")

graph["edges"].extend(new_edges)

with open("tokyo_graph_node.json", "w") as f:
    json.dump(graph, f)

print("\n=== NODE SNAPPING METRICS ===")
print("Connections:", len(new_edges))
print("Avg distance:", sum(distances)/len(distances))
print("Max distance:", max(distances))
print("Min distance:", min(distances))

print(f"\nTotal time: {time.time() - start_time:.1f} seconds")
"""

Graph loaded
Total nodes: 87091
Road nodes: 76609
POI nodes: 10482
Processing POI 0/10482 | Time: 0.4s
Processing POI 500/10482 | Time: 18.1s
Processing POI 1000/10482 | Time: 36.4s
Processing POI 1500/10482 | Time: 55.0s
Processing POI 2000/10482 | Time: 72.7s
Processing POI 2500/10482 | Time: 92.0s
Processing POI 3000/10482 | Time: 109.7s
Processing POI 3500/10482 | Time: 127.7s
Processing POI 4000/10482 | Time: 147.2s
Processing POI 4500/10482 | Time: 166.3s
Processing POI 5000/10482 | Time: 185.6s
Processing POI 5500/10482 | Time: 203.4s
Processing POI 6000/10482 | Time: 221.2s
Processing POI 6500/10482 | Time: 242.0s
Processing POI 7000/10482 | Time: 261.1s
Processing POI 7500/10482 | Time: 281.8s
Processing POI 8000/10482 | Time: 299.7s
Processing POI 8500/10482 | Time: 319.5s
Processing POI 9000/10482 | Time: 337.2s
Processing POI 9500/10482 | Time: 354.9s
Processing POI 10000/10482 | Time: 374.3s

Finished connecting POIs

=== NODE SNAPPING METRICS ===
Connections: 10482
Avg di

In [ ]:
#EDGE SNAPPING (APPROX)- brute force- WRONG
"""
import json
import math

with open("tokyo_full_graph_updated.json") as f:
    graph = json.load(f)

nodes = graph["nodes"]
edges = graph["edges"]

node_dict = {n["id"]: n for n in nodes}
poi_nodes = [n for n in nodes if n.get("type") in ["building", "hospital", "fire_station"]]

def point_to_segment_dist(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return math.sqrt((px-x1)**2 + (py-y1)**2)

    t = ((px-x1)*dx + (py-y1)*dy) / (dx*dx + dy*dy)
    t = max(0, min(1, t))

    proj_x = x1 + t*dx
    proj_y = y1 + t*dy

    return math.sqrt((px-proj_x)**2 + (py-proj_y)**2)

new_edges = []
distances = []

for poi in poi_nodes:
    px, py = poi["lon"], poi["lat"]
    best_edge = None
    min_dist = float("inf")

    for edge in edges[:5000]:  # speed limit
        n1 = node_dict.get(edge["from"])
        n2 = node_dict.get(edge["to"])
        if not n1 or not n2:
            continue

        d = point_to_segment_dist(px, py, n1["lon"], n1["lat"], n2["lon"], n2["lat"])
        if d < min_dist:
            min_dist = d
            best_edge = edge

    if best_edge:
        distances.append(min_dist)

        new_edges.append({
            "from": poi["id"],
            "to": best_edge["from"],
            "length": min_dist * 100000,
            "road_type": "connector_edge"
        })

graph["edges"].extend(new_edges)

with open("tokyo_graph_edge.json", "w") as f:
    json.dump(graph, f)

print("\n=== EDGE SNAPPING METRICS ===")
print("Connections:", len(new_edges))
print("Avg distance:", sum(distances)/len(distances))
print("Max distance:", max(distances))
print("Min distance:", min(distances))
"""


=== EDGE SNAPPING METRICS ===
Connections: 10482
Avg distance: 0.002838463353996317
Max distance: 0.02577911189335768
Min distance: 0.0
